# AICE Associate 방과후 · 4회차
## 데이터 전처리 ① — 이상치와 결측치 처리 (교사용 · 정답/해설 포함)

| 항목 | 내용 |
|---|---|
| 일시 | 11월 11일(수) 16:00~18:00 (120분) |
| 시험 영역 | **데이터 전처리 (5~9번, 30점)** 중 **5·6번 + 7번 앞부분(범주 치환·컬럼 삭제)** |
| 사용 데이터 | `signal_data.csv` (10,000행 × 12열) / `wine_quality_data.csv` (약 9만 행 × 15열) |
| 연결 | 3회차 시각화·탐색에서 찾은 이상치·결측치 → **4회차 청소** → 5회차 전처리②(원-핫 인코딩·분할·스케일링 = 7·8·9번) |

### 수업 목표
1. `drop()`의 axis=0/1, `columns=`, `inplace`, 재할당을 구분한다.
2. 조건 필터와 IQR로 이상치를 제거한다.
3. 결측치를 확인하고 삭제(dropna) 또는 대체(fillna)한다.
4. 시험 5·6번을 유형에 상관없이 해결한다.

## 👩‍🏫 수업 지도안

| 시간 | 단계 | 활동 | 비고 |
|---|---|---|---|
| 16:00~16:10 (10') | 복습 | 3회차 1~4번 **보지 않고** 다시 쓰기 (0-4 셀) | 못 쓰는 학생은 3회차 치트시트 허용 |
| 16:10~16:30 (20') | 개념 교실 | 작은 표로 `drop`·axis·조건 필터·`isnull`/`dropna`/`fillna` 익히기 | **이 20분이 오늘의 핵심.** 결과를 예상 → 실행 → 비교 |
| 16:30~16:55 (25') | Part A | 회귀 샘플 5·6번 + fillna 대안 | 두 가지 풀이(조건 필터 vs drop index) 비교 |
| 16:55~17:05 (10') | 휴식 | | |
| 17:05~17:35 (30') | Part B | 분류 샘플 5번(IQR 빈칸)·6번(빈칸)·7번 앞부분(치환·오류정정) | boxplot 그림과 fence 숫자를 연결 |
| 17:35~17:52 (17') | Part C | 시험형 연습 6문항 + 자동 채점 | ★도전 |
| 17:52~18:00 (8') | 정리 | 함정 체크리스트, 5회차 예고 | |

### 사전 준비 체크
- [ ] 3회차와 같은 데이터 2개 (학생 Colab 런타임이 끊겼으면 다시 업로드해야 함 → 수업 시작 전 미리 올리게 하기)
- [ ] 이번 회차는 **셀을 위에서부터 순서대로** 실행해야 합니다(df → df_temp → df_na로 이어짐). 중간 셀만 다시 돌리다 꼬이면 "런타임 → 모두 실행"으로 처음부터.
- [ ] Part C의 C6은 Part B 5번을 정확히 수행해야 맞는 문항(이상치 제거 후 결측치 개수)이라, 5번을 틀린 학생은 C6에서 걸러집니다.

### 수준별 운영
- **기초**: 개념 교실을 충분히(결과 예상 활동 위주), Part A·B는 빈칸형 위주, Part C는 🟢 문항(C3·C5).
- **숙련**: Part A를 두 가지 방법으로, Part C 전체 + ★도전(조건 2개 결합, 가짜 결측치 `'-'` 처리).

### 이번 회차 범위에 대한 메모
시험 전처리 영역(5~9번) 중 **7번 원-핫 인코딩(get_dummies), 8번 train_test_split, 9번 스케일링은 5회차**로 넘겼습니다(커리큘럼의 전처리①/② 구분). 오늘은 7번 문항 중 앞부분인 **범주 값 치환과 불필요 컬럼 삭제**까지만 다룹니다.

---
## 0. 준비

### 0-1. (Colab 전용) 한글 폰트 설치

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.font_manager as fm
for path in fm.findSystemFonts():
    if 'Nanum' in path:
        fm.fontManager.addfont(path)
print('나눔 폰트 등록 완료')

### 0-2. 공통 실행 코드 (시험지 제공 코드)

In [ ]:
import platform
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

os_name = platform.system()
if os_name == 'Linux':
    plt.rc('font', family='NanumGothicCoding')
elif os_name == 'Windows':
    font = fm.FontProperties(fname='C:\\Windows\\Fonts\\gulim.ttc').get_name()
    mpl.rc('font', family=font)
elif os_name == 'Darwin':
    plt.rc('font', family='AppleGothic')
mpl.rcParams['axes.unicode_minus'] = False
print('준비 완료')

### 0-3. 🔁 복습: 3회차 1~4번 보지 않고 쓰기 (10분) 🟢
아래 4가지를 **보지 않고** 써 보세요. 막히면 3회차 치트시트를 봐도 돼요.
1. 내비게이션 데이터를 `df`에 읽고 첫 4행 출력
2. 주소1(Address1)의 countplot
3. 실주행시간(x)–평균시속(y) jointplot
4. 와인 데이터를 `data`에 읽고, 등급(x)별 와이너리 평점(y) boxplot

In [ ]:
# 복습 1~4 정답
df = pd.read_csv('./signal_data.csv')
display(df.head(4))

sns.countplot(data=df, x='Address1')
plt.show()

sns.jointplot(data=df, x='Time_Driving', y='Speed_Per_Hour')
plt.show()

data = pd.read_csv('./wine_quality_data.csv')
sns.boxplot(data=data, x='Grade', y='FE_points_winery')
plt.show()

#### 👩‍🏫 복습 포인트
- jointplot 오른쪽 위의 점 1개(시속 약 305), boxplot 수염 밖의 점들을 다시 손가락으로 짚게 한 뒤 "오늘 이걸 지웁니다"로 바로 연결하세요.
- 이 셀에서 `df`, `data`가 만들어지므로 **이후 셀은 이 셀을 실행했다는 전제**입니다.

---
## 📋 시험의 '데이터 전처리' 영역 (5~9번, 30점)

| 번호 | 회귀 샘플 (내비게이션) | 분류 샘플 (와인) | 오늘? |
|---|---|---|---|
| 5 | 평균시속 ≥ 300 행 삭제 + RID 열 삭제 → `df_temp` | IQR로 와이너리 평점 이상치 삭제 + E_title 열 삭제 + 인덱스 초기화 → `data_temp` (빈칸) | ✅ 오늘 |
| 6 | 결측치 개수 → `답안6_1`, 결측 행 삭제 → `df_na` | 결측치 확인·삭제 → `data_na` (빈칸) | ✅ 오늘 |
| 7 | 시간 컬럼 삭제 + 원-핫 인코딩 (빈칸) | **와인종류 값 치환** + 컬럼 5개 삭제 + 원-핫 인코딩 (빈칸 + 오류정정) | ◐ 치환·삭제만 오늘 |
| 8 | train_test_split (80:20) | train_test_split (70:30, stratify) | 5회차 |
| 9 | RobustScaler (오류정정) | StandardScaler (오류정정) | 5회차 |

> 📌 **전처리 문항은 이어달리기예요.** `df → df_temp → df_na → df_preset`처럼 앞 문항의 결과 변수를 다음 문항이 그대로 씁니다. 5번에서 변수명을 틀리면 6번부터 줄줄이 무너져요. **변수명은 문제에 적힌 그대로!**

---
# 🧪 개념 교실 — 작은 표로 먼저 익히기 (20분)

진짜 데이터는 1만 행이라 결과가 눈에 안 들어와요. **5행짜리 표**로 먼저 연습하고 시험 문제로 가요.
각 셀을 실행하기 **전에** 결과를 먼저 예상해 보세요!

In [ ]:
mini = pd.DataFrame({
    'id':    ['a', 'b', 'c', 'd', 'e'],
    'speed': [30, 45, 305, 25, 50],
    'dist':  [3000, np.nan, 5000, 4000, np.nan],
    'city':  ['서울', '경기', '경기', '-', '서울']
})
mini

### ① drop — 행을 지울까, 열을 지울까?
| 쓰는 법 | 의미 |
|---|---|
| `mini.drop('id', axis=1)` | **열** 'id' 삭제 (axis=1 = 열 방향) |
| `mini.drop(columns=['id'])` | 위와 같음 (axis 안 써도 됨) |
| `mini.drop(2, axis=0)` | **행** 인덱스 2 삭제 (axis=0 = 행, 생략하면 기본값 0) |
| `mini.drop([0, 2])` | 행 인덱스 0, 2 삭제 |

> 🧠 외우는 법: **axis=0은 세로로 쭉 (행들을 따라), axis=1은 가로로 쭉 (열들을 따라)**. 헷갈리면 "**1**열"로 기억!

In [ ]:
# 예상해 보기: 결과 표에 몇 행, 몇 열이 남을까?
mini.drop('id', axis=1)

In [ ]:
mini.drop(2, axis=0)

### ② 원본이 바뀔까? — 재할당 vs inplace
`drop()`은 **원본을 바꾸지 않고 새 표를 돌려줘요.** 결과를 남기려면 둘 중 하나:
```python
mini2 = mini.drop('id', axis=1)          # 방법 1: 새 변수에 받기 (재할당)
mini.drop('id', axis=1, inplace=True)    # 방법 2: 원본을 직접 수정
```
⚠️ **절대 금지**: `mini = mini.drop('id', axis=1, inplace=True)` → inplace=True는 아무것도 돌려주지 않아서(**None**) mini가 사라져요!

In [ ]:
# 실험: 함정에 빠지면 어떻게 될까?
test = mini.copy()
test = test.drop('id', axis=1, inplace=True)
print(test)    # 예상해 보기!

### ③ 조건 필터 — 원하는 행만 남기기
`mini[조건]` → 조건이 True인 행만 남아요. **지우고 싶은 것의 반대**를 조건으로 쓰는 게 핵심!
- 시속 300 **이상을 지우고 싶다** → 300 **미만을 남긴다**: `mini[mini['speed'] < 300]`
- 조건 2개: `&`(그리고), `|`(또는), **각 조건을 괄호로** 감싸야 해요.

In [ ]:
print(mini['speed'] < 300)        # True/False 목록
mini[mini['speed'] < 300]

In [ ]:
# 조건 2개: 시속 300 미만 '그리고' 도시가 '-'가 아닌 행
mini[(mini['speed'] < 300) & (mini['city'] != '-')]

### ④ 결측치 — 세고, 지우고, 채우기
| 코드 | 의미 |
|---|---|
| `mini.isnull()` | 칸마다 결측이면 True |
| `mini.isnull().sum()` | **열별** 결측치 개수 |
| `mini.isnull().sum().sum()` | 전체 결측치 개수 (숫자 하나) |
| `mini.dropna()` | 결측치가 **하나라도 있는 행** 삭제 |
| `mini['dist'].fillna(값)` | 결측치를 값으로 채우기 |

> `isnull()`과 `isna()`는 완전히 같은 함수예요. 시험 빈칸에는 보통 `isnull`이 나와요.

In [ ]:
mini.isnull().sum()

In [ ]:
print('전체 결측치:', mini.isnull().sum().sum())
mini.dropna()

In [ ]:
# 채우기: 평균으로? 중앙값으로?
mini_fill = mini.copy()
mini_fill['dist'] = mini_fill['dist'].fillna(mini_fill['dist'].median())
mini_fill

#### ✍️ 개념 체크 (정답)
1. **KeyError**. axis 기본값이 0(행)이라 'speed'라는 **행 인덱스**를 찾다가 없어서 에러. → `axis=1` 또는 `columns=`.
2. **안 잡힘.** `'-'`는 글자일 뿐 NaN이 아님. `value_counts()`로 봐야 보임 → 필요하면 `replace('-', np.nan)` 후 처리.
3. `mode()`는 값 하나가 아니라 **Series(목록)** 를 돌려줘서 인덱스끼리 엇갈려 채워지지 않음(에러 없이 조용히 실패!). → `mode()[0]`으로 첫 번째 값을 꺼내야 함.

> 수업 팁: 1번은 학생들이 실제로 실행해 에러 메시지를 읽게 하세요. "KeyError: ... not found in axis"를 보면 axis 문제라는 걸 스스로 떠올리는 연습이 오류정정형 문항에 그대로 쓰입니다.

---
# Part A. 회귀 샘플 — 내비게이션 데이터 청소

> 이 Part는 0-3 복습 셀에서 만든 `df`를 그대로 사용해요. 아직 안 했다면 아래 셀부터!

In [ ]:
df = pd.read_csv('./signal_data.csv')
print(df.shape)

### A-1. [시험 5번] 이상치 행 + 불필요 열 삭제 🟢
> jointplot으로 확인한 결과, 평균시속(Speed_Per_Hour) 컬럼의 이상치를 처리하려고 합니다.
> - 대상 데이터프레임 : df
> - 평균시속이 **300보다 크거나 같은** 이상치가 있는 행(row)을 삭제하세요.
> - AI모델링에 불필요한 경로ID(**RID**) 컬럼을 삭제하세요.
> - 전처리를 적용한 데이터프레임은 **df_temp**라는 변수명으로 저장하세요.

💡 "300 이상을 **지운다**" = "300 **미만을 남긴다**" 로 바꿔 생각하기

In [ ]:
# [5번 정답] 방법 1 — 조건 필터 (샘플 정답 코드와 같은 방식)
df_temp = df[df.Speed_Per_Hour < 300]
df_temp.drop('RID', axis=1, inplace=True)

print(df_temp.shape)   # (9999, 11)

#### 👩‍🏫 해설
- 결과: 10,000행 → **9,999행**(1행 삭제), 12열 → **11열**.
- `df.Speed_Per_Hour`와 `df['Speed_Per_Hour']`는 같음. 컬럼명에 공백·한글이 있으면 대괄호만 가능하므로 대괄호를 기본으로 가르치는 것도 방법.
- **방법 2 (drop + index)** — 분류 샘플 5번과 같은 패턴이라 숙련 학생에게 함께 보여 주세요:
  ```python
  df_temp = df.drop(df[df.Speed_Per_Hour >= 300].index, axis=0)
  df_temp = df_temp.drop('RID', axis=1)
  ```
  지울 행을 조건으로 **골라서** `.index`(행 번호)만 꺼낸 뒤 drop에 넘기는 구조.
- 흔한 오답: `> 300`(같음 포함 안 함 — 이 데이터에선 결과가 같지만 문제 문구는 '크거나 같은'), `df_temp = df_temp.drop('RID', axis=1, inplace=True)`(None), `drop('RID')`(axis 누락 → KeyError).
- Colab의 pandas 2.x에서는 조건 필터 결과에 inplace drop을 하면 `SettingWithCopyWarning`이 뜰 수 있는데, 공통 코드의 `warnings.filterwarnings('ignore')`로 가려지고 결과도 정상입니다. 학생이 물으면 "필터한 표에 바로 수정하면 pandas가 원본이 바뀌는지 헷갈려서 내는 경고"라고만 설명하면 충분.

### A-2. [시험 6-1번] 결측치 개수 확인 🟢
> - 대상 데이터프레임 : df_temp
> - 결측치 개수를 확인하는 코드를 작성하세요.
> - 발견된 결측치의 개수를 **'답안6_1'** 변수에 저장해 주세요. (예시: `답안6_0 = 1234`)

In [ ]:
# [6-1번 정답]
print('결측치 처리전\n', df_temp.isnull().sum())

답안6_1 = 2      # Distance 2개 (df_temp.isnull().sum().sum()으로도 확인 가능)

### A-3. [시험 6-2번] 결측치 행 삭제 🟢
> - 대상 데이터프레임 : df_temp
> - 결측치가 있는 행을 모두 삭제하세요.
> - 전처리가 반영된 데이터프레임은 변수명 **df_na**에 저장하세요.
> - 결측치가 사라졌는지 확인하세요.

In [ ]:
# [6-2번 정답]
df_na = df_temp.dropna()
print('\n결측치 처리후\n', df_na.isnull().sum())
print(df_na.shape)    # (9997, 11)

#### 👩‍🏫 해설
- "결측치가 사라졌는지 **확인**하세요" → 확인 코드(`isnull().sum()`)까지 써야 완전한 답. 학생들이 자주 빼먹습니다.
- 답안6_1은 **숫자**로: `답안6_1 = '2'`(문자열)는 채점 위험. 예시 `답안6_0 = 1234`의 따옴표 유무를 보고 판단하라고 가르치세요(숫자 예시엔 따옴표 없음, 이름 예시엔 따옴표 있음).

### A-4. 다른 방법: 지우지 않고 채우기 (fillna) 🟢
시험에서 "결측치를 **평균/중앙값/최빈값으로 대체**하세요"라고 나올 수도 있어요. 숫자 컬럼은 평균·중앙값, 글자 컬럼은 최빈값을 주로 써요.

In [ ]:
df_fill = df_temp.copy()
df_fill['Distance'] = df_fill['Distance'].fillna(df_fill['Distance'].median())
print(df_fill['Distance'].isnull().sum())   # 0

# 글자 컬럼을 최빈값으로 채울 때는 반드시 [0]
# df_fill['Address1'] = df_fill['Address1'].fillna(df_fill['Address1'].mode()[0])

#### 👩‍🏫 해설 — 지울까 채울까?
- 지우기(dropna): 결측 행이 **아주 적을 때**(여기선 1만 개 중 2개). 간단하고 왜곡 없음.
- 채우기(fillna): 결측이 많아 지우면 데이터가 크게 줄 때. 평균은 이상치에 끌려가므로 **꼬리가 긴 분포는 중앙값**이 안전 (Distance 평균 7,024 vs 중앙값 6,121).
- 시험은 "어떻게 처리할지"를 문제에서 지정하므로 판단보다 **문구를 정확히 따르는 것**이 중요하다고 강조.
- `fillna(..., inplace=True)`를 열 하나에 쓰는 방식(`df['A'].fillna(0, inplace=True)`)은 최신 pandas에서 원본에 반영되지 않을 수 있으니 **재할당 방식**으로 가르치는 것을 권장.

### A-5. (미리보기) 7번 앞부분 — 모델에 쓸 수 없는 열 지우기 🟢
> 원-핫 인코딩 전, **'Time_Departure', 'Time_Arrival'** 컬럼을 삭제해 주세요. (대상: df_na)

출발·도착 '시각' 문자열은 그대로는 모델이 못 써요(게다가 도착시각은 정답을 알려주는 반칙 정보!). **여러 열을 한 번에** 지울 땐 리스트로.

In [ ]:
df_na.drop(['Time_Departure', 'Time_Arrival'], axis=1, inplace=True)
df_na.info()     # 9개 열 남음: 남은 object 컬럼은 Address1, Address2 → 5회차 원-핫 인코딩 대상

---
# Part B. 분류 샘플 — 와인 데이터 청소

In [ ]:
data = pd.read_csv('./wine_quality_data.csv')
print(data.shape)

### B-1. IQR로 이상치 기준선(fence) 구하기 — 개념 🟢
boxplot의 수염 끝이 바로 이 기준선이에요.

```
lower_fence = Q1 − 1.5 × IQR          upper_fence = Q3 + 1.5 × IQR
         ●  │──────┌────┬────┐──────│  ●
     이상치  lower   Q1  중앙값  Q3   upper  이상치
                    └── IQR ──┘
```
- `quantile(0.25)` = Q1 (아래에서 25% 지점), `quantile(0.75)` = Q3

In [ ]:
q1 = data['FE_points_winery'].quantile(0.25)
q3 = data['FE_points_winery'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
print('Q1:', q1, ' Q3:', q3, ' IQR:', iqr)
print('lower_fence:', lower_fence, ' upper_fence:', upper_fence)

📌 3회차 boxplot(등급별)과 숫자가 조금 다를 수 있어요. 시험 5번은 **등급을 나누지 않고 전체** 와이너리 평점으로 기준선을 구하거든요.

### B-2. [시험 5번] 이상치 행 삭제 + 열 삭제 + 인덱스 초기화 (빈칸형) 🟢
> - 대상 데이터프레임 : data
> - 와이너리 평점이 upper_fence보다 **큰** / lower_fence보다 **작은** 이상치 행을 삭제하세요.
> - AI모델링에 불필요한 와인이름(**E_title**) 컬럼을 삭제하세요.
> - 전처리를 적용한 데이터프레임은 **data_temp**로 저장하고, 완료한 뒤 **index를 초기화**해 주세요.
> - `< blank5-1 >`, `< blank5-2 >`에 들어갈 내용을 '답안5_1', '답안5_2'에 저장하세요.

In [ ]:
# [5번 정답]
q1 = data['FE_points_winery'].quantile(0.25)
q3 = data['FE_points_winery'].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

data_temp = data.drop( data[(data['FE_points_winery'] > upper_fence) | (data['FE_points_winery'] < lower_fence)].index, axis=0)
data_temp = data_temp.drop( columns=['E_title'])
data_temp.reset_index(drop=True, inplace=True)

답안5_1 = 'drop'
답안5_2 = 'index'

print(data.shape, '→', data_temp.shape)

#### 🔍 코드 한 줄 해부
```python
data.drop(  data[ (조건1) | (조건2) ].index  , axis=0)
           └──── 지울 행들만 골라낸 표 ────┘
                                     └ 그 행들의 번호 ┘
└ 그 번호의 행을 지워라 ┘
```
- `|` = 또는: 위로 벗어났거나 **또는** 아래로 벗어난 행
- `reset_index(drop=True)`: 행을 지우면 번호가 0,1,3,7...처럼 듬성듬성 → 0부터 다시 매김. `drop=True`가 없으면 옛 번호가 **새 열(index)로 추가**돼 버려요.

#### 👩‍🏫 해설
- 빈칸 오답 유형: `'dropna'`(결측치 아님), `'index()'`(괄호 없음 — index는 함수가 아니라 속성), `'loc'`.
- 조건 필터로 반대로 남기는 방식도 같은 결과: `data[(data['FE_points_winery'] <= upper_fence) & (data['FE_points_winery'] >= lower_fence)]` → **부등호와 &/|가 모두 뒤집힌다**는 점(드모르간)이 숙련 학생용 좋은 질문.
- 실행 결과의 행 수(약 9만 → 감소)는 학생들이 직접 출력해 확인하게 하세요. 정확한 숫자를 미리 말하지 않는 게 '실행해서 확인하는 습관'에 좋습니다.

### B-3. [시험 6번] 결측치 확인 → 삭제 (빈칸형) 🟢
> - 대상 데이터프레임 : data_temp
> - 결측치 개수를 확인한 후에 결측치가 있는 행을 모두 삭제하세요.
> - 전처리가 반영된 데이터프레임은 **data_na**에 저장하고, 결측치가 사라졌는지 확인하세요.
> - `< blank6-1 >`, `< blank6-2 >`에 들어갈 내용을 '답안6_1', '답안6_2'에 저장하세요.

In [ ]:
# [6번 정답]
print( '결측치 처리 전\n', data_temp.isnull().sum() )
data_na = data_temp.dropna()
print( '결측치 처리 후\n', data_na.isnull().sum() )

답안6_1 = 'isnull'
답안6_2 = 'dropna'

#### 👩‍🏫 해설
- 샘플 정답 실행 결과(처리 전): **E_FE_wine_kind 3,746 / E_price 6,902**, 나머지 0. → Part C의 C6 정답 근거.
- `isna`도 기능상 같지만, 빈칸이 `isnull`로 설계된 샘플이므로 `isnull`로 통일해서 가르치세요.

### B-4. [시험 7번 앞부분] 와인종류 값 정리 — isin + loc 🟢
3회차 countplot에서 와인종류가 `white|sparkling`, `red|rose|sparkling` 등 12가지로 너무 지저분했죠? 4가지(red / white / rose / sparkling)로 정리해요.

> - 대상 데이터프레임 : data_na
> - `'white|rose|sparkling'`, `'red|white|sparkling'`, `'red|white|rose|sparkling'`, `'red|sparkling'` → **'sparkling'**
> - `'white|sparkling'` → **'white'**
> - `'red|rose'`, `'red|white'`, `'red|rose|sparkling'` → **'red'**
> - 치환 후 와인종류의 카테고리별 개수를 출력하세요.

📌 `isin([목록])` = 값이 목록 안에 **있으면 True**. `loc[행 조건, 열] = 새 값` = 조건에 맞는 행의 그 열만 바꿔요.

In [ ]:
# 먼저 작은 예로 확인
s = pd.Series(['red', 'red|rose', 'white', 'red|white'])
print(s.isin(['red|rose', 'red|white']))

In [ ]:
# [7번 앞부분 정답]
data_na.loc[data_na['E_FE_wine_kind'].isin(['white|rose|sparkling','red|white|sparkling', 'red|white|rose|sparkling', 'red|sparkling']), ['E_FE_wine_kind']] = 'sparkling'
data_na.loc[data_na['E_FE_wine_kind'].isin(['white|sparkling']), ['E_FE_wine_kind']] = 'white'
data_na.loc[data_na['E_FE_wine_kind'].isin(['red|rose', 'red|white', 'red|rose|sparkling']), ['E_FE_wine_kind']] = 'red'
print(data_na['E_FE_wine_kind'].value_counts())

답안7 = 42262     # 샘플 정답: red 42262 / white 29126 / sparkling 5626 / rose 2910

### B-5. [시험 7번 앞부분] 불필요 컬럼 5개 삭제 (오류 정정형) 🟢
> 불필요한 컬럼 5개('E_province', 'E_region_1', 'E_region_2', 'E_winery', 'E_wine_variety')를 삭제하세요.
> 아래 코드에는 오류가 **1군데** 있어요.

In [ ]:
# [정답] axis=0 → axis=1 (컬럼 삭제이므로)
data_na = data_na.drop( ['E_province', 'E_region_1', 'E_region_2', 'E_winery', 'E_wine_variety'], axis=1)
data_na.info()     # 남은 object 컬럼: Grade, E_country, E_FE_wine_kind → 5회차에서 E_country, E_FE_wine_kind 원-핫 인코딩

#### 👩‍🏫 해설
- 실제 샘플 7번은 이 뒤에 `pd.one_hot_encoding` → `pd.get_dummies` 오류 정정이 하나 더 있습니다(5회차에서 다룸).
- `isin` 대신 `replace`로도 치환 가능: `data_na['E_FE_wine_kind'].replace(['white|sparkling'], 'white')`. 다만 빈칸형 문항이 `isin`으로 설계되어 있으니 `isin + loc` 패턴을 꼭 익히게 하세요.
- `loc[행, ['E_FE_wine_kind']]`처럼 열을 리스트로 줘도, `'E_FE_wine_kind'`처럼 문자열로 줘도 결과는 같습니다.

---
# Part C. 시험형 연습 문제 (자동 채점)

채점 셀을 먼저 실행하세요. `채점('C1', 답안C1)`처럼 쓰면 돼요.

In [ ]:
# 채점 셀 — 실행만 하세요
import hashlib
_정답 = {"C1": "aab3238922bcc25a6f606eb525ffdc56", "C2_1": "978d76676f5e7918f81d28e7d092ca0d", "C2_2": "a666587afda6e89aec274a3657558a27", "C3": "d3d9446802a44259755d38e6d163e820", "C4": "03fcd68e5673f08be96d2b6bb5be8261", "C5_1": "0d9959e8150fc305808d8838e888b622", "C5_2": "38ad6fe465c71d3a30b0d1a94766e8b0", "C6": "e2459ae2cf7921589685a6835ec08a7c"}
def 채점(번호, 답):
    h = hashlib.md5(str(답).strip().replace(' ', '').lower().encode()).hexdigest()
    print(f'[{번호}]', '⭕ 정답!' if h == _정답.get(번호) else '❌ 다시 생각해 보세요')
print('채점 준비 완료')

### C1. (직접 작성형) 🟢
> 대상 데이터프레임 : **df** (원본)
> 평균시속(Speed_Per_Hour)이 **100보다 크거나 같은** 행은 몇 개인지 구해 '답안C1'에 숫자로 저장하세요.
> 💡 조건 필터 결과의 `shape[0]` 또는 `len()`

In [ ]:
# [C1 정답]
print(len(df[df['Speed_Per_Hour'] >= 100]))
# 또는 (df['Speed_Per_Hour'] >= 100).sum()  → True를 1로 세어 줌

답안C1 = 14
채점('C1', 답안C1)

### C2. (직접 작성형 — IQR) ★
> 대상 데이터프레임 : **df** (원본)
> 실주행시간(Time_Driving)에 IQR 방법을 적용하려고 합니다.
> - upper_fence(Q3 + 1.5×IQR)를 구하고, **반올림한 정수**를 '답안C2_1'에 저장하세요. (`round()` 사용)
> - Time_Driving이 upper_fence보다 **큰** 행의 개수를 '답안C2_2'에 저장하세요.

In [ ]:
# [C2 정답]
q1 = df['Time_Driving'].quantile(0.25)
q3 = df['Time_Driving'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(upper, (df['Time_Driving'] > upper).sum())

답안C2_1 = round(upper)     # 2099  (Q1≈601, Q3≈1200, IQR≈599)
답안C2_2 = 326
채점('C2_1', 답안C2_1)
채점('C2_2', 답안C2_2)

# 생각해 볼 점: 326개는 정말 '오류'일까? 오래 걸린 주행일 뿐일 수도 → 시험은 지시한 컬럼·기준만 처리

### C3. (오류 정정형) 🟢
> 원본 df에서 RID와 Time_Departure 컬럼을 삭제한 df_c를 만들고, 결측치 개수를 확인하려 합니다. 오류 **2군데**를 고친 뒤, df_c의 **컬럼 개수**를 '답안C3'에 저장하세요.

In [ ]:
# [C3 정답] ① axis=0 → axis=1   ② isnull → isnull()
df_c = df.drop(['RID', 'Time_Departure'], axis=1)
print(df_c.isnull().sum())
print(df_c.shape)       # (10000, 10)

답안C3 = 10
채점('C3', 답안C3)

### C4. (직접 작성형 — fillna) 🟢
> 원본 df를 복사한 df_f에서 Distance 컬럼의 결측치를 **Distance의 중앙값**으로 채우세요.
> 채운 값(중앙값)을 정수로 '답안C4'에 저장하세요.

In [ ]:
# [C4 정답]
df_f = df.copy()
med = df_f['Distance'].median()
df_f['Distance'] = df_f['Distance'].fillna(med)
print(med, df_f['Distance'].isnull().sum())

답안C4 = 6121
채점('C4', 답안C4)

### C5. (빈칸 — 와인) 🟢
> B-4에서 와인종류 값을 바꿀 때 쓴 빈칸 두 개, `< blank7-1 >`과 `< blank7-2 >`에 들어간 함수 이름을 각각 '답안C5_1', '답안C5_2'에 저장하세요. (괄호 없이 이름만)

In [ ]:
# [C5 정답]
답안C5_1 = 'isin'
답안C5_2 = 'value_counts'
채점('C5_1', 답안C5_1)
채점('C5_2', 답안C5_2)

### C6. (결과 확인 — 와인) 🟢
> B-3의 **결측치 처리 전** 출력에서, 가격(**E_price**) 컬럼의 결측치 개수를 '답안C6'에 저장하세요.
> (B-2의 5번을 정확히 풀어야 맞출 수 있어요!)

In [ ]:
# [C6 정답]
print(data_temp['E_price'].isnull().sum())
답안C6 = 6902
채점('C6', 답안C6)

### ★ 도전 과제
1. 내비게이션 Address1의 **가짜 결측치 `'-'`** 를 `replace('-', np.nan)`으로 진짜 결측치로 바꾼 뒤, `isnull().sum()`으로 몇 개인지 확인하고 삭제해 보세요.
2. 원본 df에서 **평균시속 300 미만이면서 거리(Distance)가 결측이 아닌** 행만 한 줄로 남겨 보세요. (힌트: `&`, `notnull()`)
3. C5의 isin+loc 치환을 **`replace`** 로 다시 써 보세요.

In [ ]:
# ★ 도전 과제 예시 답안
# 1.
df_x = df.copy()
df_x['Address1'] = df_x['Address1'].replace('-', np.nan)
print(df_x['Address1'].isnull().sum())    # 3
df_x = df_x.dropna(subset=['Address1'])
print(df_x.shape)                          # (9997, 12)

# 2.
df_y = df[(df['Speed_Per_Hour'] < 300) & (df['Distance'].notnull())]
print(df_y.shape)                          # (9997, 12)

# 3.
w = data_temp['E_FE_wine_kind'].replace(
    {'white|rose|sparkling': 'sparkling', 'red|white|sparkling': 'sparkling',
     'red|white|rose|sparkling': 'sparkling', 'red|sparkling': 'sparkling',
     'white|sparkling': 'white',
     'red|rose': 'red', 'red|white': 'red', 'red|rose|sparkling': 'red'})
print(w.value_counts())

---
# 📝 오늘의 정리

### 전처리 ① 치트시트
| 하고 싶은 것 | 코드 |
|---|---|
| 열 삭제 | `df.drop('열', axis=1)` · `df.drop(columns=['열1', '열2'])` |
| 행 삭제 (번호로) | `df.drop(행번호들, axis=0)` |
| 조건으로 이상치 행 삭제 | `df[df['열'] < 기준]` (남길 것을 조건으로) |
| 조건으로 이상치 행 삭제 (drop 방식) | `df.drop(df[지울조건].index, axis=0)` |
| IQR 기준선 | `q1 = quantile(0.25)`, `q3 = quantile(0.75)`, `iqr = q3-q1`, `q1-1.5*iqr`, `q3+1.5*iqr` |
| 인덱스 다시 매기기 | `df.reset_index(drop=True, inplace=True)` |
| 결측치 개수 | `df.isnull().sum()` (열별) · `.sum().sum()` (전체) |
| 결측 행 삭제 | `df.dropna()` |
| 결측치 채우기 | `df['열'] = df['열'].fillna(df['열'].median())` · 최빈값은 `mode()[0]` |
| 값 바꾸기 | `df.loc[df['열'].isin([...]), '열'] = '새값'` |
| 값별 개수 | `df['열'].value_counts()` |

### 시험 함정 체크리스트
- [ ] 열을 지울 땐 **axis=1** (또는 `columns=`)
- [ ] `inplace=True`를 쓰면 **앞에 `변수 =`를 붙이지 않는다**
- [ ] 조건 2개는 `(조건1) & (조건2)`, **괄호 필수**, `and`/`or` 말고 `&`/`|`
- [ ] "300 이상 삭제" = "300 **미만** 남기기"
- [ ] `reset_index(drop=True)` — drop=True를 빼먹지 않는다
- [ ] `isnull()`, `index` — 괄호가 필요한 것(함수)과 아닌 것(속성) 구분
- [ ] 문제가 말한 **변수명**(df_temp, df_na, data_temp, data_na)으로 저장
- [ ] "확인하세요"라고 하면 **확인 코드까지** 작성

### 다음 시간 (5회차, 11/16 월) 예고 — 전처리 ②
오늘 만든 `df_na`, `data_na`를 이어서 **원-핫 인코딩(get_dummies) → train/valid 분리(train_test_split) → 스케일링**까지 하면 모델에 넣을 준비 끝! 시험 7·8·9번이에요.

#### 👩‍🏫 마무리·다음 회차 연결
- 5회차 시작 10분 복습: 회귀 샘플 5·6번을 **보지 않고** 쓰기 → 바로 7번 get_dummies로 연결.
- 공식 문서 찾기 미니 챌린지(1~2분): pandas 사이트에서 `DataFrame.drop` 문서를 찾아 `axis`의 기본값과 `columns` 파라미터 설명 읽기.
- 오늘 Part C에서 C2(IQR)와 C3(axis)을 틀린 학생 이름을 메모해 두었다가 5회차 짝 배치에 반영하세요.